# M0 · 03 — Shapes: reshape, view, transpose

Deep learning is mostly *rearranging* the same numbers into different shapes.
The GPT does this in two crucial spots:
- `logits.view(B*T, C)` — flatten batch+time together for the loss.
- `k.transpose(-2, -1)` — swap the last two axes so `q @ kᵀ` lines up.

**reshape/view** keep the numbers in order but regroup them; **transpose**
actually swaps axes (reordering how numbers are laid out logically).

In [ ]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

## 1. reshape — same numbers, new shape

`x.reshape(rows, cols)` regroups the elements *in order*. The total count must
match (`rows*cols == numel`).

Reshape `torch.arange(6)` into a **2×3** matrix (`[[0,1,2],[3,4,5]]`).

In [ ]:
x = torch.arange(6)
m = TODO
m

In [ ]:
check('reshape 2x3', m, torch.tensor([[0, 1, 2], [3, 4, 5]]))

## 2. `-1` means 'infer this axis'

Pass `-1` for one axis and PyTorch computes it. `x.reshape(3, -1)` → 3 rows,
cols inferred. Handy when you know one dimension but not the other.

Reshape the same `x` into **3 rows** with columns inferred.

In [ ]:
m2 = TODO
m2

In [ ]:
check('reshape 3x?', m2, torch.tensor([[0, 1], [2, 3], [4, 5]]))

## 3. Flatten batch+time — the GPT's `view(B*T, C)`

Before the loss, logits `(B, T, C)` are flattened to `(B*T, C)` so every
position becomes an independent classification example.

Flatten `logits` below from `(2, 3, 4)` to `(6, 4)`.

In [ ]:
logits = torch.arange(2 * 3 * 4).reshape(2, 3, 4)  # (B,T,C)
flat = TODO
flat.shape

In [ ]:
check_tensor('flattened to (6,4)', flat, shape=(6, 4))

## 4. view vs reshape (quick note, nothing to fill in)

`view` is like `reshape` but only works on *contiguous* memory (it never copies).
`reshape` will copy if needed. The GPT uses `view` because its tensors are
contiguous there. For you: **prefer `reshape` unless you know you need `view`.**
Run the cell to see they agree here.

In [ ]:
a = torch.arange(6)
print('view :', a.view(2, 3))
print('reshape:', a.reshape(2, 3))
check('view == reshape here', a.view(2, 3), a.reshape(2, 3))

## 5. transpose — swap two axes

`m.transpose(0, 1)` swaps rows/cols of a 2-D tensor. Unlike reshape, this
changes which number sits where.

Transpose the 2×3 matrix `m` into a **3×2** matrix.

In [ ]:
m = torch.tensor([[0, 1, 2], [3, 4, 5]])
mt = TODO
mt

In [ ]:
check('transposed', mt, torch.tensor([[0, 3], [1, 4], [2, 5]]))

## 6. The GPT's `k.transpose(-2, -1)`

For a batched tensor `(B, T, hs)`, we swap the **last two** axes (T and hs) to get
`(B, hs, T)`, so it can be matmul'd with `q`. Negative axes count from the end:
`-1` = last, `-2` = second-to-last — this way it works no matter the batch size.

Transpose `k` of shape `(2, 3, 4)` to shape `(2, 4, 3)`.

In [ ]:
k = torch.randn(2, 3, 4)  # (B, T, hs)
kt = TODO
kt.shape

In [ ]:
check_tensor('k transposed to (2,4,3)', kt, shape=(2, 4, 3))

## ✅ Recap

- `reshape`/`view` regroup the same numbers in order; `-1` infers one axis.
- `view(B*T, C)` flattens batch+time for the loss.
- `transpose(a, b)` swaps two axes (changes layout); `k.transpose(-2,-1)` makes
  `q @ kᵀ` line up regardless of batch size.

Next: **04 — elementwise ops & broadcasting** (how `tok_emb + pos_emb` works).